# MongoDB Setup Notebook

Use this notebook once to create the MongoDB collections, validators, and indexes required by the app.

## Prerequisites

1. Make sure `.env.local` contains `MONGODB_URI`.
2. Install `pymongo` in the Python environment used by Jupyter:

```bash
python3 -m pip install pymongo
```

In [14]:
from __future__ import annotations

import json
import os
from pathlib import Path
from urllib.parse import urlparse

from pymongo import ASCENDING, DESCENDING, MongoClient
from pymongo.errors import CollectionInvalid, OperationFailure

In [15]:
ROOT_DIR = Path.cwd()
if not (ROOT_DIR / '.env.local').exists() and (ROOT_DIR.parent / '.env.local').exists():
    ROOT_DIR = ROOT_DIR.parent

ENV_FILE = ROOT_DIR / '.env.local'
DEFAULT_DB_NAME = 'black_scholes_app'

# Optional notebook overrides
URI_OVERRIDE = None
DB_NAME_OVERRIDE = None
DROP_EXISTING = False

In [16]:
CALCULATIONS_VALIDATOR = {
    '$jsonSchema': {
        'bsonType': 'object',
        'required': ['authUserId', 'inputs', 'results'],
        'properties': {
            'authUserId': {'bsonType': 'string'},
            'userEmail': {'bsonType': 'string'},
            'inputs': {
                'bsonType': 'object',
                'required': ['spotPrice', 'strikePrice', 'timeToExpiry', 'riskFreeRate', 'volatility', 'optionType'],
                'properties': {
                    'spotPrice': {'bsonType': ['double', 'int', 'long', 'decimal']},
                    'strikePrice': {'bsonType': ['double', 'int', 'long', 'decimal']},
                    'timeToExpiry': {'bsonType': ['double', 'int', 'long', 'decimal']},
                    'riskFreeRate': {'bsonType': ['double', 'int', 'long', 'decimal']},
                    'volatility': {'bsonType': ['double', 'int', 'long', 'decimal']},
                    'optionType': {'enum': ['call', 'put']},
                },
            },
            'results': {
                'bsonType': 'object',
                'required': ['prices', 'greeks'],
                'properties': {
                    'prices': {
                        'bsonType': 'object',
                        'required': ['call', 'put'],
                        'properties': {
                            'call': {'bsonType': ['double', 'int', 'long', 'decimal']},
                            'put': {'bsonType': ['double', 'int', 'long', 'decimal']},
                        },
                    },
                    'greeks': {
                        'bsonType': 'object',
                        'required': ['deltaCall', 'deltaPut', 'gamma', 'thetaCall', 'thetaPut', 'vega', 'rhoCall', 'rhoPut'],
                        'properties': {
                            'deltaCall': {'bsonType': ['double', 'int', 'long', 'decimal']},
                            'deltaPut': {'bsonType': ['double', 'int', 'long', 'decimal']},
                            'gamma': {'bsonType': ['double', 'int', 'long', 'decimal']},
                            'thetaCall': {'bsonType': ['double', 'int', 'long', 'decimal']},
                            'thetaPut': {'bsonType': ['double', 'int', 'long', 'decimal']},
                            'vega': {'bsonType': ['double', 'int', 'long', 'decimal']},
                            'rhoCall': {'bsonType': ['double', 'int', 'long', 'decimal']},
                            'rhoPut': {'bsonType': ['double', 'int', 'long', 'decimal']},
                        },
                    },
                },
            },
            'createdAt': {'bsonType': 'date'},
            'updatedAt': {'bsonType': 'date'},
        },
    }
}

USER_PROFILES_VALIDATOR = {
    '$jsonSchema': {
        'bsonType': 'object',
        'required': ['authUserId', 'email', 'name', 'image', 'angelOne'],
        'properties': {
            'authUserId': {'bsonType': 'string'},
            'email': {'bsonType': 'string'},
            'name': {'bsonType': 'string'},
            'image': {'bsonType': 'string'},
            'angelOne': {
                'bsonType': 'object',
                'required': ['apiKey', 'clientCode', 'clientLocalIp', 'clientPublicIp', 'macAddress', 'session'],
                'properties': {
                    'apiKey': {'bsonType': 'string'},
                    'clientCode': {'bsonType': 'string'},
                    'clientLocalIp': {'bsonType': 'string'},
                    'clientPublicIp': {'bsonType': 'string'},
                    'macAddress': {'bsonType': 'string'},
                    'session': {
                        'bsonType': 'object',
                        'required': ['jwtToken', 'refreshToken', 'feedToken', 'loggedInAt', 'status', 'errorMessage'],
                        'properties': {
                            'jwtToken': {'bsonType': 'string'},
                            'refreshToken': {'bsonType': 'string'},
                            'feedToken': {'bsonType': 'string'},
                            'loggedInAt': {'bsonType': 'string'},
                            'lastConnectedAt': {'bsonType': ['date', 'null']},
                            'status': {'enum': ['connected', 'disconnected', 'error']},
                            'errorMessage': {'bsonType': 'string'},
                        },
                    },
                },
            },
            'createdAt': {'bsonType': 'date'},
            'updatedAt': {'bsonType': 'date'},
        },
    }
}

In [17]:
def read_env_file(path: Path) -> dict[str, str]:
    if not path.exists():
        return {}

    values: dict[str, str] = {}
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def resolve_uri() -> str:
    if URI_OVERRIDE:
        return URI_OVERRIDE
    env_values = read_env_file(ENV_FILE)
    uri = env_values.get('MONGODB_URI') or os.environ.get('MONGODB_URI', '')
    if not uri or 'your_mongodb_connection_string_here' in uri:
        raise RuntimeError('MONGODB_URI is not configured. Update .env.local or set URI_OVERRIDE.')
    return uri


def resolve_db_name(uri: str) -> str:
    if DB_NAME_OVERRIDE:
        return DB_NAME_OVERRIDE
    env_values = read_env_file(ENV_FILE)
    env_db_name = env_values.get('MONGODB_DB_NAME') or os.environ.get('MONGODB_DB_NAME')
    if env_db_name:
        return env_db_name
    parsed = urlparse(uri)
    path_name = parsed.path.lstrip('/')
    if path_name:
        return path_name.split('?')[0]
    return DEFAULT_DB_NAME


def create_or_update_collection(db, name: str, validator: dict):
    existing = db.list_collection_names()
    if name not in existing:
        try:
            db.create_collection(name, validator=validator, validationLevel='moderate')
            print(f'Created collection: {name}')
            return
        except CollectionInvalid:
            pass
    db.command({'collMod': name, 'validator': validator, 'validationLevel': 'moderate'})
    print(f'Updated validator: {name}')


def ensure_indexes(db):
    calculations = db['calculations']
    userprofiles = db['userprofiles']

    calculations.create_index([('authUserId', ASCENDING), ('createdAt', DESCENDING)], name='authUserId_createdAt')
    calculations.create_index([('userEmail', ASCENDING)], name='userEmail')

    userprofiles.create_index([('authUserId', ASCENDING)], name='authUserId_unique', unique=True)
    userprofiles.create_index([('email', ASCENDING)], name='email')

    print('Ensured indexes for calculations and userprofiles.')


def maybe_drop_existing(db):
    for name in ('calculations', 'userprofiles'):
        if name in db.list_collection_names():
            db[name].drop()
            print(f'Dropped collection: {name}')

In [18]:
uri = resolve_uri()
db_name = resolve_db_name(uri)

client = MongoClient(uri)
db = client[db_name]

print({'env_file': str(ENV_FILE), 'database': db_name, 'drop_existing': DROP_EXISTING})

{'env_file': '/Users/manishsm/Desktop/Personal/personal_projects/BlackScholesOptionPricing/.env.local', 'database': 'black_scholes_app', 'drop_existing': False}


## Execute setup

Run the next cell once you have reviewed the target database name above.

In [19]:
if DROP_EXISTING:
    maybe_drop_existing(db)

create_or_update_collection(db, 'calculations', CALCULATIONS_VALIDATOR)
create_or_update_collection(db, 'userprofiles', USER_PROFILES_VALIDATOR)
ensure_indexes(db)

summary = {
    'database': db_name,
    'collections': sorted(db.list_collection_names()),
    'indexes': {
        'calculations': list(db['calculations'].index_information().keys()),
        'userprofiles': list(db['userprofiles'].index_information().keys()),
    },
}
print(json.dumps(summary, indent=2))

OperationFailure: user is not allowed to do action [collMod] on [black_scholes_app.calculations], full error: {'ok': 0, 'errmsg': 'user is not allowed to do action [collMod] on [black_scholes_app.calculations]', 'code': 8000, 'codeName': 'AtlasError'}